# Binary TMJ Position Classifier — Training (A+B)

3D CNN для **бинарной** классификации положения головок ВНЧС: **central vs non-central**.  
Подход A: детектор-кропы 128³ (точный ROI вместо центральных кропов).  
Подход B: BinaryFocalLoss + калибровка порога по ROC / Youden's J.

**Поддерживаемые среды:** Yandex DataSphere · Google Colab · локально

---

Полная документация: `MLService/docs/superpowers/specs/2026-04-10-binary-classifier-improvement-design.md`

## 1. Setup

In [1]:
# Установка зависимостей
# Без кавычек — DataSphere %pip не поддерживает quoted specs
%pip install -q --upgrade scipy tqdm nibabel scikit-learn


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
# pydicom не нужен в этом ноутбуке:
# кропы (.nii.gz) грузятся через nibabel, не через pydicom.
# Генерация кропов (секция 3) запускается как subprocess.
import sys

In [3]:
import os
from pathlib import Path

# --- Определение среды ---
IN_DATASPHERE = Path("/home/jupyter").exists()
IN_COLAB = "google.colab" in sys.modules
IN_LOCAL = not IN_DATASPHERE and not IN_COLAB

env_name = "DataSphere" if IN_DATASPHERE else ("Colab" if IN_COLAB else "Local")
print(f"Среда: {env_name}")

# --- Пути ---
if IN_DATASPHERE:
    # Репо клонировано в проект DataSphere
    REPO_ROOT = Path("/home/jupyter/project/MLService")
    # Данные: датасет tmj_data в DataSphere
    # Вариант 1 — датасет через #pragma: /home/jupyter/datasets/tmj_data/
    # Вариант 2 — загружен вручную и unzip'нут:
    DATA_ROOT = Path("/home/jupyter/datasets/tmj_data")
    WORK_ROOT = Path("/home/jupyter/project")

elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/MLService")
    DATA_ROOT = Path("/content/drive/MyDrive/tmj_data")
    WORK_ROOT = Path("/content")

else:  # Local
    REPO_ROOT = Path(".").resolve().parent  # MLService/
    DATA_ROOT = REPO_ROOT / "data"
    WORK_ROOT = REPO_ROOT

# --- Добавить корень репо в sys.path для импортов ---
sys.path.insert(0, str(REPO_ROOT))

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"DATA_ROOT : {DATA_ROOT}")
print(f"Exists    : {DATA_ROOT.exists()}")

Среда: DataSphere
REPO_ROOT : /home/jupyter/project/MLService
DATA_ROOT : /home/jupyter/datasets/tmj_data
Exists    : True


In [4]:
# Загрузка detector model с локальной машины → DataSphere
# Запусти эту ячейку один раз, если DETECTOR_PATH exists=False
# /home/jupyter/datasets/ read-only — сохраняем в project/
import os
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets

dest_dir = Path("/home/jupyter/project/models")
dest_dir.mkdir(parents=True, exist_ok=True)
dest_path = dest_dir / "best_detector.pth"

if dest_path.exists():
    print(f"Детектор уже есть: {dest_path}")
else:
    upload = widgets.FileUpload(accept=".pth", multiple=False)
    btn = widgets.Button(description="Сохранить")
    out = widgets.Output()

    def on_save(b):
        with out:
            if not upload.value:
                print("Сначала выбери файл")
                return
            content = list(upload.value.values())[0]["content"]
            dest_path.write_bytes(content)
            print(f"Сохранено: {dest_path} ({len(content)/1e6:.1f} MB)")

    btn.on_click(on_save)
    print("Выбери файл best_model.pth с локальной машины → нажми Сохранить")
    print("(файл: MLService/experiments/detector_20251126_003305/best_model.pth)")
    display(widgets.VBox([upload, btn, out]))

OSError: [Errno 30] Read-only file system: '/home/jupyter/datasets/tmj_data/models'

In [ ]:
# --- Конфигурация путей к данным ---

DATASET_ROOT    = DATA_ROOT / "dataset_public"          # папка со study_XXXX/
MANIFEST_PATH   = DATASET_ROOT / "manifest_private.json"
LABELS_PATH     = DATA_ROOT / "tmj_position_labels.json"
DETECTOR_PATH   = Path("/home/jupyter/project/models/best_detector.pth")

CROPS_DIR       = WORK_ROOT / "data" / "detector_crops"  # куда сохранять NIfTI кропы
OUTPUT_DIR      = WORK_ROOT / "experiments"

CROPS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATASET_ROOT  : {DATASET_ROOT}  exists={DATASET_ROOT.exists()}")
print(f"MANIFEST_PATH : {MANIFEST_PATH}  exists={MANIFEST_PATH.exists()}")
print(f"LABELS_PATH   : {LABELS_PATH}  exists={LABELS_PATH.exists()}")
print(f"DETECTOR_PATH : {DETECTOR_PATH}  exists={DETECTOR_PATH.exists()}")
print(f"CROPS_DIR     : {CROPS_DIR}")
print(f"OUTPUT_DIR    : {OUTPUT_DIR}")

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS (Apple Silicon)")
else:
    device = torch.device("cpu")
    print("CPU")

print(f"PyTorch: {torch.__version__}")

## 2. Label Table

Загрузка меток из `manifest_private.json` + `tmj_position_labels.json`.  
Бинаризация: **class 0 (central) → 0**, **classes 1 и 2 → 1 (non-central)**.

In [ ]:
from training.tmj_position_label_table import build_index, binarize_labels, split_by_patient

all_records = build_index(
    manifest_path=str(MANIFEST_PATH),
    labels_path=str(LABELS_PATH),
    dataset_root=str(DATASET_ROOT),
)
print(f"Всего записей: {len(all_records)}")

binary_records = binarize_labels(all_records, crop_dir=str(CROPS_DIR))
print(f"Бинарных записей (×2 стороны): {len(binary_records)}")

from collections import Counter
sag_dist = Counter(r["sag"] for r in binary_records)
fr_dist  = Counter(r["fr"]  for r in binary_records)
print(f"Sagittal — 0 (central): {sag_dist[0]}, 1 (non-central): {sag_dist[1]}")
print(f"Frontal  — 0 (central): {fr_dist[0]},  1 (non-central): {fr_dist[1]}")

In [ ]:
# Разбивка train / val строго по пациенту (без утечки данных)
SPLIT_RATIO = 0.8
train_records, val_records = split_by_patient(binary_records, split_ratio=SPLIT_RATIO, seed=42)

print(f"Train: {len(train_records)} кропов")
print(f"Val:   {len(val_records)} кропов")

## 3. Preprocessing: Detector → Crops (один раз)

Запускать **один раз**. Детектор находит центры левого/правого ВНЧС,  
вырезает 128³ куб в оригинальном пространстве и сохраняет как `.nii.gz`.

Если кропы уже есть в `CROPS_DIR` — пропустить эту секцию.

In [ ]:
# Проверить сколько кропов уже есть
existing_crops = list(CROPS_DIR.rglob("*.nii.gz"))
print(f"Кропов найдено: {len(existing_crops)}")
print(f"Ожидается: {len(all_records) * 2} ({len(all_records)} исследований × 2 стороны)")

NEED_CROPS = len(existing_crops) < len(all_records) * 2
print(f"\nГенерировать кропы: {NEED_CROPS}")

In [ ]:
if NEED_CROPS:
    if not DETECTOR_PATH.exists():
        raise FileNotFoundError(
            f"Детектор не найден: {DETECTOR_PATH}\n"
            "Положите best_detector.pth в DATA_ROOT/models/ и перезапустите."
        )

    # Используем существующий инструмент из репо
    import subprocess, sys
    cmd = [
        sys.executable,
        str(REPO_ROOT / "tools" / "auto_crop_from_detector.py"),
        "--model",   str(DETECTOR_PATH),
        "--input",   str(DATASET_ROOT),
        "--output",  str(CROPS_DIR),
        "--crop_size", "128",
        "--batch",
        "--format",  "nifti",
        "--device",  str(device).split(":")[0],  # cuda / mps / cpu
    ]
    print("Запуск:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(REPO_ROOT), capture_output=False)
    print("Return code:", result.returncode)
else:
    print("Кропы уже есть — пропускаем генерацию.")

In [ ]:
# Проверка: обновить crop_path в binary_records на реальные пути
# (binarize_labels уже строит пути на основе CROPS_DIR — они должны совпасть)
missing = [r for r in binary_records if not Path(r["crop_path"]).exists()]
print(f"Кропов не найдено: {len(missing)} из {len(binary_records)}")
if missing:
    print("Примеры:", [r["crop_path"] for r in missing[:3]])

## 4. Dataset & DataLoader

In [ ]:
from torch.utils.data import DataLoader
from training.datasets.tmj_position_dataset import TMJBinaryPositionDataset

BATCH_SIZE  = 4
NUM_WORKERS = 0  # 0 в DataSphere / Colab; можно увеличить локально

train_ds = TMJBinaryPositionDataset(train_records, is_train=True)
val_ds   = TMJBinaryPositionDataset(val_records,   is_train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train: {len(train_ds)} сэмплов, {len(train_loader)} батчей")
print(f"Val:   {len(val_ds)} сэмплов, {len(val_loader)} батчей")

In [ ]:
# Быстрая проверка — один батч
vol, lbl = next(iter(train_loader))
print(f"volume: {vol.shape} dtype={vol.dtype}  min={vol.min():.3f} max={vol.max():.3f}")
print(f"labels: {lbl.shape} dtype={lbl.dtype}")
print(f"labels sample: {lbl}")
print(f"  [sag, fr] — 0=central, 1=non-central")

## 5. Model

**TMJBinaryPositionClassifier** — 3D CNN с двумя бинарными головами:  
- `head_sag` → sagittal position (central / non-central)  
- `head_fr`  → frontal position (central / non-central)

Каждая голова выдаёт 1 логит. Решение: `sigmoid(logit) > threshold`.

In [ ]:
from models.tmj_binary_position_classifier import TMJBinaryPositionClassifier

model = TMJBinaryPositionClassifier().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Параметров: {n_params / 1e6:.2f}M")

# Smoke test
with torch.no_grad():
    dummy = torch.randn(2, 1, 32, 48, 48).to(device)
    sag_logit, fr_logit = model(dummy)
print(f"sag_logit: {sag_logit.shape}, fr_logit: {fr_logit.shape}")

## 6. Training

In [ ]:
import datetime
import json
import numpy as np
import torch.optim as optim
from tqdm.notebook import tqdm
from training.losses.focal_loss import BinaryFocalLoss

# --- Гиперпараметры ---
EPOCHS         = 100
LR             = 1e-4
WEIGHT_DECAY   = 1e-5
LR_PATIENCE    = 10
EARLY_STOPPING = 30
GAMMA          = 2.0   # focal loss gamma

HEAD_NAMES = ["sag", "fr"]

In [ ]:
# --- Автовычисление alpha из распределения классов ---
def compute_class_weights(loader):
    counts = {name: {0: 0, 1: 0} for name in HEAD_NAMES}
    for _, labels in loader:
        for i, name in enumerate(HEAD_NAMES):
            for cls in (0, 1):
                counts[name][cls] += (labels[:, i] == cls).sum().item()
    alphas = {}
    for name in HEAD_NAMES:
        total = counts[name][0] + counts[name][1]
        alphas[name] = counts[name][0] / total if total > 0 else 0.5
        print(f"[{name}] 0={counts[name][0]} ({100*counts[name][0]/max(total,1):.1f}%)"
              f"  1={counts[name][1]} ({100*counts[name][1]/max(total,1):.1f}%)"
              f"  → alpha={alphas[name]:.3f}")
    return alphas

alphas = compute_class_weights(train_loader)

In [ ]:
# --- Loss, оптимизатор, scheduler ---
criteria = {
    name: BinaryFocalLoss(gamma=GAMMA, alpha=alphas[name])
    for name in HEAD_NAMES
}

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=LR_PATIENCE
)

# --- Эксперимент ---
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
exp_dir = OUTPUT_DIR / f"binary_position_{timestamp}"
exp_dir.mkdir(parents=True, exist_ok=True)

config = {
    "epochs": EPOCHS, "lr": LR, "weight_decay": WEIGHT_DECAY,
    "gamma": GAMMA, "alphas": alphas, "batch_size": BATCH_SIZE,
    "split_ratio": SPLIT_RATIO, "train_samples": len(train_ds),
    "val_samples": len(val_ds), "timestamp": timestamp,
}
with open(exp_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Эксперимент: {exp_dir}")
print(f"Config: {config}")

In [ ]:
# --- Вспомогательные функции ---

def compute_metrics(sag_logit, fr_logit, labels, thresholds=(0.5, 0.5)):
    metrics = {}
    for i, (name, logit, thresh) in enumerate(
        zip(HEAD_NAMES, [sag_logit, fr_logit], thresholds)
    ):
        probs = torch.sigmoid(logit.squeeze(1))
        preds = (probs >= thresh).long()
        acc   = (preds == labels[:, i]).float().mean().item()
        metrics[f"acc_{name}"] = acc
    metrics["mean_accuracy"] = float(np.mean([metrics[f"acc_{n}"] for n in HEAD_NAMES]))
    return metrics


def train_epoch(epoch):
    model.train()
    running_loss, all_m = 0.0, []
    for volumes, labels in tqdm(train_loader, desc=f"Epoch {epoch} [Train]", leave=False):
        volumes, labels = volumes.to(device), labels.to(device)
        optimizer.zero_grad()
        sag_logit, fr_logit = model(volumes)
        loss = (
            criteria["sag"](sag_logit, labels[:, 0].float()) +
            criteria["fr"](fr_logit,  labels[:, 1].float())
        )
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            m = compute_metrics(sag_logit, fr_logit, labels)
        m["loss"] = loss.item()
        all_m.append(m)
        running_loss += loss.item()
    avg = {k: float(np.mean([m[k] for m in all_m])) for k in all_m[0]}
    avg["loss"] = running_loss / len(train_loader)
    return avg


def val_epoch(epoch):
    model.eval()
    running_loss, all_m = 0.0, []
    with torch.no_grad():
        for volumes, labels in tqdm(val_loader, desc=f"Epoch {epoch} [Val]  ", leave=False):
            volumes, labels = volumes.to(device), labels.to(device)
            sag_logit, fr_logit = model(volumes)
            loss = (
                criteria["sag"](sag_logit, labels[:, 0].float()) +
                criteria["fr"](fr_logit,  labels[:, 1].float())
            )
            m = compute_metrics(sag_logit, fr_logit, labels)
            m["loss"] = loss.item()
            all_m.append(m)
            running_loss += loss.item()
    avg = {k: float(np.mean([m[k] for m in all_m])) for k in all_m[0]}
    avg["loss"] = running_loss / len(val_loader)
    return avg

In [ ]:
# --- Цикл обучения ---
best_val_acc   = -1.0
epochs_no_imp  = 0
history        = []
best_model_path = exp_dir / "best_model.pth"

for epoch in range(1, EPOCHS + 1):
    train_m = train_epoch(epoch)
    val_m   = val_epoch(epoch)
    scheduler.step(val_m["mean_accuracy"])
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"[{epoch:3d}/{EPOCHS}] "
        f"train loss={train_m['loss']:.4f} acc={train_m['mean_accuracy']:.3f} "
        f"(sag={train_m['acc_sag']:.3f} fr={train_m['acc_fr']:.3f}) | "
        f"val loss={val_m['loss']:.4f} acc={val_m['mean_accuracy']:.3f} "
        f"(sag={val_m['acc_sag']:.3f} fr={val_m['acc_fr']:.3f}) "
        f"lr={current_lr:.6f}"
    )

    row = {"epoch": epoch, "lr": current_lr}
    row.update({f"train_{k}": v for k, v in train_m.items()})
    row.update({f"val_{k}": v   for k, v in val_m.items()})
    history.append(row)
    with open(exp_dir / "metrics.jsonl", "a") as f:
        f.write(json.dumps(row) + "\n")

    if val_m["mean_accuracy"] > best_val_acc:
        best_val_acc = val_m["mean_accuracy"]
        epochs_no_imp = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_accuracy": best_val_acc,
            "val_metrics": val_m,
        }, best_model_path)
        print(f"  ✓ Сохранён best model (acc={best_val_acc:.3f})")
    else:
        epochs_no_imp += 1
        if EARLY_STOPPING > 0 and epochs_no_imp >= EARLY_STOPPING:
            print(f"  Early stopping на эпохе {epoch}")
            break

print(f"\nОбучение завершено. Best val acc: {best_val_acc:.3f}")

## 7. Threshold Calibration

После обучения: найти оптимальный порог по Youden's J = sensitivity + specificity - 1.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Загрузить best model
ckpt = torch.load(best_model_path, map_location=device, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Загружена модель с эпохи {ckpt['epoch']}, val acc={ckpt['best_val_accuracy']:.3f}")

# Собрать предсказания на val
all_probs  = {name: [] for name in HEAD_NAMES}
all_labels = {name: [] for name in HEAD_NAMES}

with torch.no_grad():
    for volumes, labels in val_loader:
        volumes = volumes.to(device)
        sag_logit, fr_logit = model(volumes)
        for i, name in enumerate(HEAD_NAMES):
            logit = [sag_logit, fr_logit][i]
            all_probs[name].extend(torch.sigmoid(logit.squeeze(1)).cpu().tolist())
            all_labels[name].extend(labels[:, i].tolist())

# Калибровка порога
calibration = {"optimal_thresholds": {}, "auc_roc": {}, "accuracy_at_threshold": {}}

for name in HEAD_NAMES:
    probs  = np.array(all_probs[name])
    labels = np.array(all_labels[name])

    if len(np.unique(labels)) < 2:
        print(f"[{name}] Только один класс в val — порог=0.5, AUC=N/A")
        calibration["optimal_thresholds"][name] = 0.5
        calibration["auc_roc"][name] = None
        calibration["accuracy_at_threshold"][name] = float(np.mean((probs >= 0.5) == labels))
        continue

    fpr, tpr, thresholds = roc_curve(labels, probs)
    auc = roc_auc_score(labels, probs)
    j_scores = tpr - fpr
    best_idx   = int(np.argmax(j_scores))
    best_thresh = float(thresholds[best_idx])
    acc = float(np.mean((probs >= best_thresh) == labels))

    calibration["optimal_thresholds"][name]     = round(best_thresh, 4)
    calibration["auc_roc"][name]                = round(auc, 4)
    calibration["accuracy_at_threshold"][name]  = round(acc, 4)

    print(f"[{name}] AUC={auc:.3f}  best_thresh={best_thresh:.3f}  acc@thresh={acc:.3f}")

# Сохранить в config.json
with open(exp_dir / "config.json") as f:
    saved_config = json.load(f)
saved_config.update(calibration)
saved_config["best_val_accuracy"] = best_val_acc
with open(exp_dir / "config.json", "w") as f:
    json.dump(saved_config, f, indent=2)

print(f"\nОптимальные пороги: {calibration['optimal_thresholds']}")
print(f"AUC-ROC:            {calibration['auc_roc']}")

## 8. Visualization

In [ ]:
import matplotlib.pyplot as plt

epochs_list = [h["epoch"] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(epochs_list, [h["train_loss"] for h in history], label="Train")
axes[0].plot(epochs_list, [h["val_loss"]   for h in history], label="Val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Mean accuracy
axes[1].plot(epochs_list, [h["train_mean_accuracy"] for h in history], label="Train")
axes[1].plot(epochs_list, [h["val_mean_accuracy"]   for h in history], label="Val")
axes[1].axhline(0.68, color="red", linestyle="--", alpha=0.5, label="Baseline v5 (0.68)")
axes[1].set_title("Mean Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Per-head val accuracy
axes[2].plot(epochs_list, [h["val_acc_sag"] for h in history], label="Sag")
axes[2].plot(epochs_list, [h["val_acc_fr"]  for h in history], label="Fr")
axes[2].set_title("Val Accuracy per Head")
axes[2].set_xlabel("Epoch")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(exp_dir / "training_curves.png", dpi=150)
plt.show()

In [ ]:
# ROC curves
from sklearn.metrics import roc_curve, auc as sklearn_auc

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, name in zip(axes, HEAD_NAMES):
    probs_arr  = np.array(all_probs[name])
    labels_arr = np.array(all_labels[name])

    if len(np.unique(labels_arr)) < 2:
        ax.text(0.5, 0.5, "Один класс в val\nROC недоступна",
                ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"{name}")
        continue

    fpr, tpr, _ = roc_curve(labels_arr, probs_arr)
    roc_auc = sklearn_auc(fpr, tpr)
    opt_thresh = calibration["optimal_thresholds"][name]

    ax.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax.set_title(f"{name.upper()} — threshold={opt_thresh}")
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(exp_dir / "roc_curves.png", dpi=150)
plt.show()

In [ ]:
# Confusion matrices с оптимальными порогами
from sklearn.metrics import confusion_matrix, classification_report

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, name in zip(axes, HEAD_NAMES):
    probs_arr  = np.array(all_probs[name])
    labels_arr = np.array(all_labels[name])
    thresh     = calibration["optimal_thresholds"][name]
    preds      = (probs_arr >= thresh).astype(int)

    cm = confusion_matrix(labels_arr, preds, labels=[0, 1])
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["central", "non-central"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["central", "non-central"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{name.upper()} (thresh={thresh})")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")

    print(f"\n=== {name.upper()} ===")
    print(classification_report(labels_arr, preds, labels=[0, 1],
                                 target_names=["central", "non-central"]))

plt.tight_layout()
fig.savefig(exp_dir / "confusion_matrices.png", dpi=150)
plt.show()

## 9. Export

In [ ]:
# Сохранить полный analysis JSON
analysis = {
    "config": saved_config,
    "total_epochs": len(history),
    "best_epoch": ckpt["epoch"],
    "best_val_accuracy": best_val_acc,
    "calibration": calibration,
    "history": history,
}
analysis_path = exp_dir / "training_analysis.json"
with open(analysis_path, "w") as f:
    json.dump(analysis, f, indent=2, ensure_ascii=False)
print(f"Analysis сохранён: {analysis_path}")

In [ ]:
# DataSphere: скопировать best_model.pth в датасет проекта (если нужно)
if IN_DATASPHERE:
    import shutil
    dest = Path("/home/jupyter/datasets/tmj_data/models")
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(best_model_path, dest / "best_binary_classifier.pth")
    shutil.copy(exp_dir / "config.json", dest / "best_binary_classifier_config.json")
    print(f"Модель скопирована в {dest}")

# Colab: скачать
elif IN_COLAB:
    from google.colab import files
    files.download(str(best_model_path))
    files.download(str(analysis_path))

# Локально: просто путь
else:
    print(f"best_model.pth : {best_model_path}")
    print(f"analysis.json  : {analysis_path}")

In [ ]:
# Итоговое резюме
print("=" * 60)
print("РЕЗУЛЬТАТЫ")
print("=" * 60)
print(f"Best val mean accuracy : {best_val_acc:.3f}  (baseline v5: 0.680)")
print(f"Эпоха best checkpoint  : {ckpt['epoch']}")
print()
for name in HEAD_NAMES:
    auc_val = calibration['auc_roc'][name]
    thresh  = calibration['optimal_thresholds'][name]
    acc_t   = calibration['accuracy_at_threshold'][name]
    print(f"  [{name}] AUC={auc_val}  thresh={thresh}  acc@thresh={acc_t}")
print()
print(f"Артефакты: {exp_dir}")